In [1]:
import os
import copy
import glob
import pandas as pd
import numpy as np
import torch
import math
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import f1_score, balanced_accuracy_score
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(device)

cuda


In [2]:
DATA_DIR = "/content/drive/MyDrive/data/"
OUTPUT_DIR = "/content/drive/MyDrive/models/"

MODEL_NAME  = "lexlms/legal-longformer-base"
MAX_LENGTH  = 2048
HIDDEN_SIZE = 768
SEED = 42

CONFIG = {
    "lora_r": 4,
    "lora_alpha": 8,
    "lora_dropout": 0.1,
    "hidden_dropout": 0.2,
    "head_lr_mult":   2.0,
    "lr": 5e-5,
    "batch_size": 2,
    "accum_steps": 8,
    "epochs": 10,
    "early_stop": 3,
    "warmup_ratio": 0.1
}


torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
def load_data(path):
    paths = [path] if os.path.isfile(path) else glob.glob(os.path.join(path, "*.csv"))
    df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    df = df[df["label"] != -1].copy()
    df = df[df["side_addressed"].isin([0.0, 1.0])].copy()
    df["label"] = df["label"].astype(int)
    df["side_addressed"] = df["side_addressed"].astype(int)
    return df

def build_case_df(df):
    cases = []
    for (case_id, justice_id), group in df.groupby(["case_id", "justice_id"]):
        pet_rows = group[group["side_addressed"] == 0].sort_values("turn_position")
        res_rows = group[group["side_addressed"] == 1].sort_values("turn_position")

        def join(rows):
            return " [SEP] ".join(
                str(r["preceding_context"]) + " </s> " + str(r["justice_utterance"])
                for _, r in rows.iterrows()
            ).strip()

        pet_text = join(pet_rows) or "[NO PETITIONER EXCHANGES]"
        res_text = join(res_rows) or "[NO RESPONDENT EXCHANGES]"

        cases.append({
            "case_id": case_id,
            "justice_id": justice_id,
            "pet_text": pet_text,
            "res_text": res_text,
            "label": group["label"].iloc[0],
        })

    return pd.DataFrame(cases)

raw_df  = load_data(DATA_DIR)
case_df = build_case_df(raw_df)
print(case_df.groupby("justice_id")["label"].agg(["count", "mean"]).round(3))

                          count   mean
justice_id                            
j__amy_coney_barrett        174  0.661
j__brett_m_kavanaugh        298  0.664
j__clarence_thomas          237  0.616
j__elena_kagan              789  0.613
j__john_g_roberts_jr       1196  0.657
j__ketanji_brown_jackson    116  0.612
j__neil_gorsuch             348  0.661
j__samuel_a_alito_jr       1064  0.603
j__sonia_sotomayor          898  0.597


In [4]:
def best_threshold(labels, probs):
    best_t, best_f1 = 0.5, -1
    for t in np.linspace(0.1, 0.9, 81):
        preds = (np.array(probs) >= t).astype(int)
        f1 = f1_score(labels, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_t, best_f1 = t, f1
    return best_t, best_f1

In [5]:
class JusticeDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.pet_enc = tokenizer(
            df["pet_text"].tolist(),
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        self.res_enc = tokenizer(
            df["res_text"].tolist(),
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        self.labels = torch.tensor(df["label"].tolist(), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "pet_input_ids": self.pet_enc["input_ids"][idx],
            "pet_attention_mask": self.pet_enc["attention_mask"][idx],
            "res_input_ids": self.res_enc["input_ids"][idx],
            "res_attention_mask": self.res_enc["attention_mask"][idx],
            "label": self.labels[idx],
        }

In [6]:
class DualEmbeddingClassifier(nn.Module):
    def __init__(self, encoder, dropout=0.2):
        super().__init__()
        self.encoder = encoder
        hidden_size = encoder.config.hidden_size

        self.pool_score = nn.Linear(hidden_size, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 2),
        )

    def attention_pool(self, input_ids, attention_mask):
      global_attention_mask = torch.zeros_like(attention_mask)
      global_attention_mask[:, 0] = 1

      out = self.encoder(
          input_ids=input_ids,
          attention_mask=attention_mask,
          global_attention_mask=global_attention_mask,
      )
      hidden = out.last_hidden_state

      scores = self.pool_score(hidden).squeeze(-1)
      scores = scores.masked_fill(attention_mask == 0, torch.finfo(scores.dtype).min)

      weights = torch.softmax(scores, dim=1)
      pooled = torch.bmm(weights.unsqueeze(1), hidden).squeeze(1)
      return pooled

    def forward(self, pet_input_ids, pet_attention_mask, res_input_ids, res_attention_mask):
        v_pet = self.attention_pool(pet_input_ids, pet_attention_mask)
        v_res = self.attention_pool(res_input_ids, res_attention_mask)

        combined = torch.cat([v_pet, v_res], dim=-1)
        return self.classifier(combined)

In [7]:
def train_epoch(model, loader, loss_fn, optimizer, scheduler, accum_steps):
    model.train()
    total_loss, preds, labels = 0.0, [], []
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        pet_ids = batch["pet_input_ids"].to(device)
        pet_mask = batch["pet_attention_mask"].to(device)
        res_ids = batch["res_input_ids"].to(device)
        res_mask = batch["res_attention_mask"].to(device)
        lbls = batch["label"].to(device)

        logits = model(pet_ids, pet_mask, res_ids, res_mask)
        loss = loss_fn(logits, lbls) / accum_steps
        loss.backward()

        if (step + 1) % accum_steps == 0 or step == len(loader) - 1:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps
        preds.extend(logits.argmax(-1).cpu().numpy())
        labels.extend(lbls.cpu().numpy())

    return total_loss / len(loader), accuracy_score(labels, preds)


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, preds, labels, probs = 0.0, [], [], []

    with torch.no_grad():
        for batch in loader:
            pet_ids = batch["pet_input_ids"].to(device)
            pet_mask = batch["pet_attention_mask"].to(device)
            res_ids = batch["res_input_ids"].to(device)
            res_mask = batch["res_attention_mask"].to(device)
            lbls = batch["label"].to(device)

            logits = model(pet_ids, pet_mask, res_ids, res_mask)
            total_loss += loss_fn(logits, lbls).item()
            preds.extend(logits.argmax(-1).cpu().numpy())
            labels.extend(lbls.cpu().numpy())
            probs.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())

    return total_loss / len(loader), accuracy_score(labels, preds), preds, labels, probs

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_encoder = AutoModel.from_pretrained(MODEL_NAME)
print(f"Hidden size: {base_encoder.config.hidden_size}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

LongformerModel LOAD REPORT from: lexlms/legal-longformer-base
Key                                | Status     | 
-----------------------------------+------------+-
lm_head.dense.weight               | UNEXPECTED | 
lm_head.layer_norm.weight          | UNEXPECTED | 
lm_head.dense.bias                 | UNEXPECTED | 
longformer.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                       | UNEXPECTED | 
lm_head.layer_norm.bias            | UNEXPECTED | 
pooler.dense.bias                  | MISSING    | 
pooler.dense.weight                | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Hidden size: 768


In [ ]:
def train_justice(justice_id, cfg):
    # Data
    justice_df = case_df[case_df["justice_id"] == justice_id].copy()
    print(f"\n{'─'*50}")
    print(f"{justice_id}  ({len(justice_df)} cases)")
    print(f"{'─'*50}")

    label_counts = justice_df["label"].value_counts()
    if (label_counts < 3).any():
        print(f"  Skipping — insufficient class examples")
        return None

    can_stratify = (label_counts >= 6).all()
    tr, tmp = train_test_split(justice_df, test_size=0.3, random_state=SEED,
                               stratify=justice_df["label"] if can_stratify else None)
    val, te = train_test_split(tmp, test_size=0.5, random_state=SEED,
                               stratify=tmp["label"] if can_stratify else None)

    cw = torch.tensor([
        len(tr) / (2 * (tr["label"] == 0).sum()),
        len(tr) / (2 * (tr["label"] == 1).sum()),
    ], dtype=torch.float).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=cw)

    train_loader = DataLoader(JusticeDataset(tr,  tokenizer), batch_size=cfg["batch_size"], shuffle=True)
    val_loader = DataLoader(JusticeDataset(val, tokenizer), batch_size=cfg["batch_size"], shuffle=False)
    test_loader = DataLoader(JusticeDataset(te,  tokenizer), batch_size=cfg["batch_size"], shuffle=False)

    # Model
    encoder = get_peft_model(copy.deepcopy(base_encoder), LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"],
        lora_dropout=cfg["lora_dropout"],
        target_modules=["query", "value"], bias="none",
    ))
    encoder.gradient_checkpointing_enable()
    encoder.enable_input_require_grads()
    model = DualEmbeddingClassifier(encoder, dropout=cfg["hidden_dropout"]).to(device)

    # Optimiser
    optimizer = AdamW([
        {"params": model.encoder.parameters(), "lr": cfg["lr"]},
        {"params": model.classifier.parameters(), "lr": cfg["lr"] * cfg["head_lr_mult"]},
        {"params": model.pool_score.parameters(), "lr": cfg["lr"] * cfg["head_lr_mult"]},
    ], weight_decay=0.01)

    total_updates = math.ceil(len(train_loader) / cfg["accum_steps"]) * cfg["epochs"]
    warmup_steps= max(1, int(total_updates * cfg["warmup_ratio"]))
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_updates)

    # Training
    save_path = os.path.join(OUTPUT_DIR, justice_id)
    best_score = -float("inf")
    no_improve = 0

    print(f"\n{'Ep':>3} {'TrLoss':>8} {'TrAcc':>7} {'VaLoss':>8} {'VaMF1':>7}")
    for epoch in range(1, cfg["epochs"] + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, loss_fn, optimizer, scheduler, cfg["accum_steps"])
        va_loss, va_acc, va_preds, va_labels, va_probs = evaluate(model, val_loader, loss_fn)
        va_macro_f1 = f1_score(va_labels, va_preds, average="macro", zero_division=0)

        flag = ""
        if va_macro_f1 > best_score:
            best_score = va_macro_f1
            no_improve = 0
            os.makedirs(save_path, exist_ok=True)
            model.encoder.save_pretrained(save_path)
            torch.save(model.classifier.state_dict(), os.path.join(save_path, "classifier.pt"))
            torch.save(model.pool_score.state_dict(), os.path.join(save_path, "pool_score.pt"))
            np.save(os.path.join(save_path, "val_probs.npy"),  np.array(va_probs))
            np.save(os.path.join(save_path, "val_labels.npy"), np.array(va_labels))
            flag = " ✓"
        else:
            no_improve += 1
            if no_improve >= cfg["early_stop"]:
                print(f"  Early stop at epoch {epoch}")
                break

        print(f"{epoch:>3} {tr_loss:>8.4f} {tr_acc:>7.4f} {va_loss:>8.4f} {va_macro_f1:>7.4f}{flag}")

    # Reload best checkpoint
    best_base = copy.deepcopy(base_encoder)
    best_encoder = get_peft_model(best_base, LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"],
        lora_dropout=cfg["lora_dropout"],
        target_modules=["query", "value"], bias="none",
    ))
    best_encoder.load_adapter(save_path, adapter_name="default")
    best_model = DualEmbeddingClassifier(best_encoder, dropout=cfg["hidden_dropout"]).to(device)
    best_model.classifier.load_state_dict(torch.load(os.path.join(save_path, "classifier.pt"), map_location=device))
    best_model.pool_score.load_state_dict(torch.load(os.path.join(save_path, "pool_score.pt"), map_location=device))

    # Tune threshold on saved val preds
    val_probs  = np.load(os.path.join(save_path, "val_probs.npy"))
    val_labels = np.load(os.path.join(save_path, "val_labels.npy"))
    threshold, val_f1 = best_threshold(val_labels, val_probs)
    print(f"\n  Best val threshold: {threshold:.2f}  (val macro F1: {val_f1:.4f})")

    # Test with tuned threshold
    _, test_acc, _, test_labels, test_probs = evaluate(best_model, test_loader, nn.CrossEntropyLoss())
    test_preds = (np.array(test_probs) >= threshold).astype(int)

    bal_acc  = balanced_accuracy_score(test_labels, test_preds)
    macro_f1 = f1_score(test_labels, test_preds, average="macro", zero_division=0)
    baseline = max(te["label"].mean(), 1 - te["label"].mean())

    print(f"Test acc:          {accuracy_score(test_labels, test_preds):.4f}")
    print(f"Balanced acc:      {bal_acc:.4f}")
    print(f"Macro F1:          {macro_f1:.4f}")
    print(f"Majority baseline: {baseline:.4f}")
    print(classification_report(test_labels, test_preds,
          target_names=["respondent", "petitioner"], zero_division=0))

    return {
        "justice_id": justice_id,
        "test_acc": accuracy_score(test_labels, test_preds),
        "bal_acc": bal_acc,
        "macro_f1": macro_f1,
        "baseline": baseline,
        "threshold": threshold,
    }

In [10]:
results = []
completed = []
for justice_id in sorted(case_df["justice_id"].unique()):
    if justice_id in completed:
        continue
    result = train_justice(justice_id, CONFIG)
    if result:
        results.append(result)
        pd.DataFrame(results).to_csv(
            os.path.join(OUTPUT_DIR, "results.csv"), index=False
        )


──────────────────────────────────────────────────
j__amy_coney_barrett  (174 cases)
──────────────────────────────────────────────────

 Ep   TrLoss   TrAcc   VaLoss   VaMF1
  1   0.7184  0.5455   0.6651  0.3953 ✓
  2   0.6740  0.6529   0.6856  0.3953
  3   0.6929  0.6694   0.6603  0.5784 ✓
  4   0.6783  0.6364   0.6695  0.5784
  5   0.6611  0.6777   0.6590  0.5784
  Early stop at epoch 6

  Best val threshold: 0.44  (val macro F1: 0.5784)
Test acc:          0.6296
Balanced acc:      0.4722
Macro F1:          0.3864
Majority baseline: 0.6667
              precision    recall  f1-score   support

  respondent       0.00      0.00      0.00         9
  petitioner       0.65      0.94      0.77        18

    accuracy                           0.63        27
   macro avg       0.33      0.47      0.39        27
weighted avg       0.44      0.63      0.52        27


──────────────────────────────────────────────────
j__brett_m_kavanaugh  (298 cases)
─────────────────────────────────────